# Exporting to QuTiP — operators, spectra, and dynamics

`fluxcharge` builds an *analytical* Hamiltonian; QuTiP has the mature engine for
**time evolution, expectation values, and Lindblad master equations**. The
bridge is `ReductionResult.to_qutip(...)`, which hands QuTiP ready-made operator
matrices — so it works for the **gyrator and quantum-phase-slip circuits scqubits
cannot represent**.

We use a **quantum-phase-slip (QPS) qubit** — a coherent phase slip shunting an
inductor, the LCG dual of the transmon. scqubits has no QPS element; fluxcharge
does.

In [ ]:
import numpy as np
import qutip as qt
import matplotlib
matplotlib.use("Agg")            # notebook: drop this line, use %matplotlib inline
import matplotlib.pyplot as plt
from fluxcharge import library

qp = library.phase_slip_qubit(charge_bias=False)
res = qp.hamiltonian()
res.H                            # symbolic Hamiltonian: -E_S cos(q) + phi^2 / 2L

## `to_qutip` returns a small bundle

Not just `H` — also each mode's operators (and, for a compact coordinate `x`,
the displacement `e^{i x}` keyed `expi_<x>`), the mode classification, and the
per-mode Hilbert dimensions.

In [ ]:
params = {"L": 1.0, "E_S": 5.0}
bundle = res.to_qutip(params)
H   = bundle["H"]
print("H is a", type(H).__name__, "with dims", bundle["dims"])
print("operators:", list(bundle["operators"]))
print("modes:", bundle["modes"])

## Spectrum — and a cross-check against fluxcharge's own solver

QuTiP diagonalizes the exported matrix; fluxcharge's native `eigenenergies`
should agree to machine precision (same operators, same basis).

In [ ]:
ev_qutip = H.eigenenergies()[:5]
ev_qutip = ev_qutip - ev_qutip[0]
ev_native = res.eigenenergies(params, n_levels=5)
ev_native = np.asarray(ev_native) - ev_native[0]
print("qutip : ", np.round(ev_qutip, 6))
print("native: ", np.round(ev_native, 6))
print("max|diff| =", np.max(np.abs(ev_qutip - ev_native)))
assert np.max(np.abs(ev_qutip - ev_native)) < 1e-9

## Now do something QuTiP is good at: driven dynamics

Project onto the lowest two levels to get an effective qubit, then drive it on
resonance and watch a Rabi oscillation. (For a real device you would keep more
levels; two is enough to see the physics.)

In [ ]:
# two lowest eigenstates -> effective qubit
evals, evecs = H.eigenstates(eigvals=2)
w01 = evals[1] - evals[0]
g, e = evecs[0], evecs[1]
P_e = e * e.dag()                        # excited-state projector

# drive 0<->1 through the flux operator; normalize by its 0-1 matrix element
n = bundle["operators"]["phi_v2"]
n01 = abs(complex(g.dag() * n * e))      # qutip returns a scalar here
Omega = 0.1 * w01                        # drive strength

t = np.linspace(0, 60, 400)
drive = [n, lambda tt, *args: 2 * Omega / max(n01, 1e-9) * np.cos(w01 * tt)]
out = qt.sesolve([H, drive], g, t, e_ops=[P_e])
print("max excited population:", round(float(np.max(out.expect[0])), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, out.expect[0])
ax.set_xlabel("time"); ax.set_ylabel("P(excited)"); ax.set_title("Resonant Rabi drive (QPS qubit)")
fig.tight_layout()
fig   # in a live notebook this renders inline

## Adding dissipation: a Lindblad master equation

Hand QuTiP a collapse operator (here T1 relaxation via the lowering operator)
and use `mesolve`. This is the payoff of the export — none of fluxcharge's own
machinery does open-system dynamics, but every QuTiP tool now applies.

In [ ]:
a = g * e.dag()                          # |0><1| lowering within the qubit subspace
gamma = 0.02
out2 = qt.mesolve(H, e, t, c_ops=[np.sqrt(gamma) * a], e_ops=[P_e])
print("population after t=60:", round(float(out2.expect[0][-1]), 3),
      "(expected ~ exp(-gamma*t) =", round(float(np.exp(-gamma * 60)), 3), ")")

**Takeaway.** `to_qutip` is the door from fluxcharge's exact symbolic/numeric
core to QuTiP's dynamics engine — and it covers the non-reciprocal (gyrator) and
phase-slip circuits that charge-basis tools can't represent.